In [1]:
with open("balen_story.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


In [2]:
raw_text

'Balendra Shah, widely known as Balen Shah, was born in Kathmandu, Nepal. \nHe studied civil engineering and later became known in Nepal as a rapper, engineer, and public figure.\n\nBefore entering politics, Balen gained popularity through rap music that discussed social issues, corruption, youth frustration, and problems within the system. \nMany young listeners connected with his honest and direct style of expression.\n\nAfter completing his engineering studies, Balen worked on urban planning and infrastructure-related projects. \nHe often spoke publicly about unmanaged city growth, poor waste management, traffic congestion, and lack of long-term planning in Kathmandu.\n\nIn 2022, Balen Shah announced that he would run for mayor of Kathmandu Metropolitan City as an independent candidate. \nAt the time, many political analysts believed it would be difficult for an independent candidate to defeat leaders from major political parties.\n\nHis campaign heavily used social media platforms 

In [3]:
def chunk_text(text,max_words=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)
    return chunks
chunks = chunk_text(raw_text)
print(f"Number of chunks: {len(chunks)}")
print(chunks[0])

Number of chunks: 5
Balendra Shah, widely known as Balen Shah, was born in Kathmandu, Nepal. He studied civil engineering and later became known in Nepal as a rapper, engineer, and public figure. Before entering politics, Balen gained popularity through rap music that discussed social issues, corruption, youth frustration, and problems within the system. Many young listeners connected with his honest and direct style of expression. After completing his engineering studies, Balen worked on urban planning and infrastructure-related projects. He often spoke publicly about unmanaged city growth, poor waste management, traffic congestion, and lack of long-term planning in Kathmandu. In 2022, Balen Shah announced


In [4]:
len(chunks)

5

In [5]:
len(chunks[0])

696

In [6]:
import requests
import numpy as np
RAG_API_KEY = "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"

In [7]:
import requests
import numpy as np

def get_embedding(text):
    url = "https://integrate.api.nvidia.com/v1/embeddings"

    headers = {
        "Authorization": f"Bearer {RAG_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "nvidia/llama-nemotron-embed-vl-1b-v2",
        "input": text,
        "input_type": "passage"   # 🔥 FIXED
    }

    response = requests.post(url, json=payload, headers=headers)


    if response.status_code != 200:
        return None

    data = response.json()

    return np.array(data["data"][0]["embedding"])


#exmple of first chunk
test_embedding = get_embedding(chunks[0])
print(test_embedding.shape)

(2048,)


In [8]:
chunks[0]

'Balendra Shah, widely known as Balen Shah, was born in Kathmandu, Nepal. He studied civil engineering and later became known in Nepal as a rapper, engineer, and public figure. Before entering politics, Balen gained popularity through rap music that discussed social issues, corruption, youth frustration, and problems within the system. Many young listeners connected with his honest and direct style of expression. After completing his engineering studies, Balen worked on urban planning and infrastructure-related projects. He often spoke publicly about unmanaged city growth, poor waste management, traffic congestion, and lack of long-term planning in Kathmandu. In 2022, Balen Shah announced'

In [9]:
len(chunks)

5

In [10]:
import faiss

In [11]:
# Validate test_embedding before accessing its shape
if test_embedding is None:
    raise ValueError("test_embedding is not initialized. Ensure get_embedding works correctly.")

dimension = test_embedding.shape[0]
index = faiss.IndexFlatL2(dimension)

chunk_mapping = []
for chunk in chunks:
    emb = get_embedding(chunk)
    if emb is None:
        raise ValueError(f"Failed to get embedding for chunk: {chunk}")
    index.add(np.array([emb]).astype("float32"))
    chunk_mapping.append(chunk)

In [12]:
faiss.write_index(index, "index.faiss")

In [13]:
def retrieve_top_k(query, k=5):
    query_embedding = get_embedding(query)
    distances, indices = index.search(np.array([query_embedding]).astype("float32"), k)
    results = []
    for idx in indices[0]:
        results.append(chunk_mapping[idx])
    return results

In [14]:
chunk_mapping

['Balendra Shah, widely known as Balen Shah, was born in Kathmandu, Nepal. He studied civil engineering and later became known in Nepal as a rapper, engineer, and public figure. Before entering politics, Balen gained popularity through rap music that discussed social issues, corruption, youth frustration, and problems within the system. Many young listeners connected with his honest and direct style of expression. After completing his engineering studies, Balen worked on urban planning and infrastructure-related projects. He often spoke publicly about unmanaged city growth, poor waste management, traffic congestion, and lack of long-term planning in Kathmandu. In 2022, Balen Shah announced',
 'that he would run for mayor of Kathmandu Metropolitan City as an independent candidate. At the time, many political analysts believed it would be difficult for an independent candidate to defeat leaders from major political parties. His campaign heavily used social media platforms such as Faceboo

In [15]:
def build_prompt(context_chunks, query):
    context = "\n".join(context_chunks)

    return f"""
You are a helpful assistant.

Use ONLY the following context to answer the question.
If answer is not in context, say "I don't know".

Context:
{context}

Question:
{query}

Answer in clear English:
"""

In [16]:
def generate_completion(prompt):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {RAG_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "meta/llama-3.1-8b-instruct",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 200,
        "temperature": 0.2,
    }

    response = requests.post(url, json=payload, headers=headers)

    data = response.json()

    # SAFE extraction (only text)
    return data["choices"][0]["message"]["content"].strip()

In [17]:
query = "Tell me who is Balen Shah?"

top_chunks = retrieve_top_k(query=query, k=5)

print(top_chunks)

prompt = build_prompt(
    context_chunks=top_chunks,
    query=query
)

response = generate_completion(prompt=prompt)

print(response)

['discussed public figures in Nepal.', 'digitized to improve transparency and public service efficiency. During interviews, Balen frequently stated that young people should participate more actively in leadership and decision-making rather than only criticizing politics from outside. His rise from rapper and engineer to mayor inspired many young Nepalese citizens who believed independent candidates could also succeed in national politics. Even after becoming mayor, public opinion about Balen remained mixed. Supporters viewed him as a bold reformer trying to modernize Kathmandu, while critics argued that some policies lacked long-term planning and consultation. Despite criticism, Balen Shah continued to remain one of the most influential and widely', 'Balendra Shah, widely known as Balen Shah, was born in Kathmandu, Nepal. He studied civil engineering and later became known in Nepal as a rapper, engineer, and public figure. Before entering politics, Balen gained popularity through rap m